In [2]:
import re
import numpy as np
import pandas as pd
import requests, sys
from BECancerResistome import beagle2vep

ENSEMBL VEP configurations


In [3]:
server = "https://rest.ensembl.org"
ext = "/vep/human/hgvs"
headers = {"Content-Type": "application/json", "Accept": "application/json"}

## Import Beagle sgRNA annotation


In [4]:
variants = pd.read_csv(f"data/BeagleCoelho/EG_collab-guides.txt", sep="\t")
variants.shape

(24752, 24)

In [15]:
variants["editor"] = (
    variants["Edit Type"].apply(lambda v: "ABE" if v == "A-G" else "CBE").values
)

In [16]:
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,sgRNA Sequence Start Pos. (global),sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor
24357,ENST00000649815.2,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000014.9,ENSG00000142208,AKT1,-,...,104776666,sense,"104776682G>A, 104776681G>A","C_4, C_5","264C>T, 265C>T","Phe88Phe, His89Tyr","Silent, Missense",NaN,NaN,CBE
4570,ENST00000263967.4,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,179234082,antisense,179234095T>C;179234096T>C;179234097T>C,A_7;A_6;A_5,2938T>C;2939T>C;2940T>C,Phe980Pro,Missense,NaN,NaN,ABE
5433,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,55146691,sense,"55146694C>T, 55146698C>T","C_4, C_8","513C>T, 517C>T","Asp171Asp, Leu173Phe","Silent, Missense",NaN,NaN,CBE
18280,ENST00000429687.8,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,20357686,antisense,"20357698T>C, 20357699T>C;20357701T>C","A_8, A_7;A_5","1614T>C, 1615T>C;1617T>C","Gly538Gly, Tyr539His","Silent, Missense",NaN,NaN,ABE
429,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,4097297,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CBE
15697,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,226390452,sense,"226390468G>A, 226390464G>A","C_4, C_8","559C>T, 563C>T","Leu187Phe, Ala188Val","Missense, Missense",NaN,NaN,CBE
189,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,4094439,sense,4094454G>A,C_5,1091C>T,Thr364Ile,Missense,NaN,NaN,CBE
1623,ENST00000262948.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,4117503,sense,4117515G>A,C_8,207C>T,Asp69Asp,Silent,NaN,NaN,CBE
8127,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,55191732,antisense,"55191744G>A, 55191748G>A","C_8, C_4","2495G>A, 2499G>A","Arg832His, Leu833Leu","Missense, Silent",NaN,NaN,CBE
7236,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,55171174,antisense,55171190T>C,A_4,1896T>C,Gly632Gly,Silent,NaN,NaN,ABE


## Format for ENSEMBL VEP input

Use the Beagle columns [input, Nucleotide Edits] and format as HGVS identifiers for ENSEMBL VEP. Examples include:

- `ENST00000618231.3:c.9G>C`
- `ENST00000471631.1:c.28_33delTCGCGG`
- `ENST00000285667.3:c.1047_1048insC`
- `5:g.140532G>C`


In [7]:
beagle2vep(variants.loc[7343])

['ENST00000275493.7:c.1940C>T;1941C>T', 'ENST00000275493.7:c.1943C>T']

In [31]:
variants["hgvs"] = [
    "-" if r["Nucleotide Edits"] is np.nan else beagle2vep(r)
    for _, r in variants.iterrows()
]
variants.sample(10)

,Input,CRISPR Enzyme,Edit Type,Edit Window,Target Taxon,Target Assembly,Target Genome Sequence,Target Gene ID,Target Gene Symbol,Target Gene Strand,...,sgRNA Orientation,Nucleotide Edits (global),Guide Edits,Nucleotide Edits,Amino Acid Edits,Mutation Category,Constraint Violations,Note,editor,hgvs
16491,ENST00000429687.8,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000014.9,ENSG00000129484,PARP2,+,...,antisense,20343669G>A,C_4,28G>A,Gly10Ser,Missense,NaN,NaN,CBE,[ENST00000429687.8:c.28G>A]
9257,ENST00000275493.7,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,sense,"55205356C>T, 55205359C>T","C_5, C_8","3372C>T, 3375C>T","His1124His, Tyr1125Tyr","Silent, Silent",NaN,NaN,CBE,"[ENST00000275493.7:c.3372C>T, ENST00000275493...."
12936,ENST00000366794.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,antisense,226363937A>G,A_5,2786+6T>C,(NC),Intron,NaN,NaN,ABE,[ENST00000366794.10:c.2786+6T>C]
13539,ENST00000366794.10,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,antisense,"226367600C>T, 226367599C>T","C_7, C_6","2286G>A, 2287G>A","Val762Val, Glu763Lys","Silent, Missense",NaN,NaN,CBE,"[ENST00000366794.10:c.2286G>A, ENST00000366794..."
14288,ENST00000366794.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000001.11,ENSG00000143799,PARP1,-,...,antisense,226379199A>G;226379198A>G,A_8;A_7,1688T>C;1689T>C,Val563Ala,Missense,NaN,NaN,ABE,[ENST00000366794.10:c.1688T>C;1689T>C]
24659,ENST00000649815.2,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000014.9,ENSG00000142208,AKT1,-,...,sense,"104780230G>A, 104780229G>A, 104780227G>A","C_4, C_5, C_7","47-14C>T, 47-13C>T, 47-11C>T","(NC), (NC), (NC)","Intron, Intron, Intron",NaN,NaN,CBE,"[ENST00000649815.2:c.47-14C>T, ENST00000649815..."
2125,ENST00000263967.4,SpyoCas9NG,C-T,4..8,9606,GRCh38,NC_000003.12,ENSG00000121879,PIK3CA,+,...,sense,179199022C>T,C_8,197C>T,Ser66Phe,Missense,poly(T):TTTT,NaN,CBE,[ENST00000263967.4:c.197C>T]
18590,ENST00000621592.8,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000008.11,ENSG00000136997,MYC,+,...,antisense,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABE,-
6886,ENST00000275493.7,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000007.14,ENSG00000146648,EGFR,+,...,sense,"55163731A>G, 55163735A>G","A_4, A_8","1632-2A>G, 1634A>G","(NC), Glu545Gly","Splice-acceptor, Missense",NaN,NaN,ABE,"[ENST00000275493.7:c.1632-2A>G, ENST0000027549..."
1046,ENST00000262948.10,SpyoCas9NG,A-G,4..8,9606,GRCh38,NC_000019.10,ENSG00000126934,MAP2K2,-,...,antisense,4101240A>G,A_6,569T>C,Ile190Thr,Missense,NaN,NaN,ABE,[ENST00000262948.10:c.569T>C]


In [34]:
variants_hgvs = [v for vs in variants["hgvs"] if vs != "-" for v in vs]

Write variants HGVS input to file


In [35]:
len(variants_hgvs)

26104

In [11]:
with open("data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt", "w") as f:
    for variant in variants_hgvs:
        f.write(f"{variant}\n")

In [13]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-VEP-HGVS-input.txt

ENST00000262948.10:c.*12C>T
ENST00000262948.10:c.*14C>T
ENST00000262948.10:c.*7C>T
ENST00000262948.10:c.*8C>T
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.1203A>G
ENST00000262948.10:c.*2A>G
ENST00000262948.10:c.*1C>T
ENST00000262948.10:c.*9G>A
ENST00000262948.10:c.*10G>A


VEP command


In [ ]:
# ./vep --af --af_1kg --af_gnomade --af_gnomadg --appris --biotype --buffer_size 500 --ccds --check_existing --distance 5000 --domains --gencode_primary --hgvs --mane --numbers --plugin OpenTargets,file=[path_to]/OTGenetics.tsv.gz --plugin CADD,snv=[path_to]/CADD_GRCh38_1.7_whole_genome_SNVs.tsv.gz,indels=[path_to]/CADD_GRCh38_1.7_InDels.tsv.gz --plugin REVEL,file=[path_to]/new_tabbed_revel_grch38.tsv.gz --plugin SpliceAI,snv=[path_to]/spliceai_scores.masked.snv.hg38.vcf.gz,indel=[path_to]/spliceai_scores.masked.indel.hg38.vcf.gz,snv_ensembl=[path_to]/spliceai_scores.raw.snv.ensembl_mane.grch38.110.vcf.gz --plugin Enformer,file=[path_to]/enformer_grch38.vcf.gz --plugin AlphaMissense,file=[path_to]/AlphaMissense_hg38.tsv.gz --plugin Blosum62 --plugin MaveDB,file=[path_to]/MaveDB_variants.tsv.gz --plugin ClinPred,file=[path_to]/ClinPred_hg38_sorted_tabbed.tsv.gz --plugin dbscSNV,[path_to]/dbscSNV1.1_GRCh38.txt.gz --plugin EVE,file=[path_to]/eve_merged.vcf.gz --plugin LOEUF,file=[path_to]/loeuf_dataset_grch38.tsv.gz,match_by=gene --plugin NMD --plugin AncestralAllele,[path_to]/homo_sapiens_ancestor_GRCh38_109.fa.gz --plugin MaxEntScan,[path_to]/maxentscan --plugin mutfunc,db=[path_to]/mutfunc_data.db,motif=1,int=1,mod=1,exp=1 --polyphen b --protein --pubmed --regulatory --show_ref_allele --sift b --species homo_sapiens --symbol --transcript_version --tsl --uniprot --uploaded_allele --var_synonyms --cache --input_file [input_data] --output_file [output_file]

In [14]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-VEP-HGVS-output.txt

#Uploaded_variation	Location	Allele	Consequence	IMPACT	SYMBOL	Gene	Feature_type	Feature	BIOTYPE	EXON	INTRON	HGVSc	HGVSp	cDNA_position	CDS_position	Protein_position	Amino_acids	Codons	Existing_variation	REF_ALLELE	UPLOADED_ALLELE	DISTANCE	STRAND	FLAGS	SYMBOL_SOURCE	HGNC_ID	MANE	MANE_SELECT	MANE_PLUS_CLINICAL	TSL	APPRIS	CCDS	ENSP	SWISSPROT	TREMBL	UNIPARC	UNIPROT_ISOFORM	SIFT	PolyPhen	DOMAINS	HGVS_OFFSET	AF	AFR_AF	AMR_AF	EAS_AF	EUR_AF	SAS_AF	gnomADe_AF	gnomADe_AFR_AF	gnomADe_AMR_AF	gnomADe_ASJ_AF	gnomADe_EAS_AF	gnomADe_FIN_AF	gnomADe_MID_AF	gnomADe_NFE_AF	gnomADe_REMAINING_AF	gnomADe_SAS_AF	gnomADg_AF	gnomADg_AFR_AF	gnomADg_AMI_AF	gnomADg_AMR_AF	gnomADg_ASJ_AF	gnomADg_EAS_AF	gnomADg_FIN_AF	gnomADg_MID_AF	gnomADg_NFE_AF	gnomADg_REMAINING_AF	gnomADg_SAS_AF	CLIN_SIG	SOMATIC	PHENO	PUBMED	VAR_SYNONYMS	MOTIF_NAME	MOTIF_POS	HIGH_INF_POS	MOTIF_SCORE_CHANGE	TRANSCRIPTION_FACTORS	OpenTargets_geneId	OpenTargets_l2g	CADD_PHRED	CADD_RAW	REVEL	SpliceAI_pred_DP_AG	SpliceAI_pred_DP_AL	SpliceAI_pred_DP_DG

Export HGVS annotated Beagle file


In [38]:
variants.to_csv(f"data/BeagleCoelho/EG_collab-guides-HGVS.txt", sep="\t", index=False)

In [39]:
!head -n 10 data/BeagleCoelho/EG_collab-guides-HGVS.txt

Input	CRISPR Enzyme	Edit Type	Edit Window	Target Taxon	Target Assembly	Target Genome Sequence	Target Gene ID	Target Gene Symbol	Target Gene Strand	Target Transcript ID	Target Domain	sgRNA Sequence	sgRNA Context Sequence	PAM Sequence	sgRNA Sequence Start Pos. (global)	sgRNA Orientation	Nucleotide Edits (global)	Guide Edits	Nucleotide Edits	Amino Acid Edits	Mutation Category	Constraint Violations	Note	editor	hgvs
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense								ABE	-
ENST00000262948.10	SpyoCas9NG	C-T	4..8	9606	GRCh38	NC_000019.10	ENSG00000126934	MAP2K2	-	ENST00000262948.10	CDS	CCGGGCTCCCTGCGTCCCGC	GTGGCCGGGCTCCCTGCGTCCCGCTGGTGA	TG	4090572	sense	4090586G>A, 4090584G>A	C_6, C_8	*12C>T, *14C>T	(NC), (NC)	UTR, UTR			CBE	['ENST00000262948.10:c.*12C>T', 'ENST00000262948.10:c.*14C>T']
ENST00000262948.10	SpyoCas9NG	A-G	4..8	9606	GRCh38	NC_000019.10	ENSG000001

## [Optional] Rest API to query a single edit


In [128]:
vep_request = {"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}

In [129]:
# dict to string
vep_request = str(vep_request).replace("'", '"')
vep_request

'{"hgvs_notations": ["ENST00000429687.8:c.1553+4A>G"]}'

In [130]:
r = requests.post(
    server + ext,
    headers=headers,
    data=vep_request,
)

if not r.ok:
    r.raise_for_status()
    sys.exit()

decoded = r.json()
print(repr(decoded))

[{'strand': 1, 'seq_region_name': '14', 'end': 20357524, 'transcript_consequences': [{'transcript_id': 'ENST00000250416', 'gene_id': 'ENSG00000129484', 'gene_symbol': 'PARP2', 'variant_allele': 'G', 'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'biotype': 'protein_coding', 'strand': 1, 'impact': 'LOW'}, {'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'variant_allele': 'G', 'impact': 'LOW', 'consequence_terms': ['splice_donor_region_variant', 'intron_variant'], 'strand': 1, 'biotype': 'protein_coding', 'gene_id': 'ENSG00000129484', 'transcript_id': 'ENST00000429687', 'gene_symbol': 'PARP2'}, {'gene_id': 'ENSG00000129484', 'transcript_id': 'ENST00000527384', 'gene_symbol': 'PARP2', 'distance': 1569, 'hgnc_id': 'HGNC:272', 'gene_symbol_source': 'HGNC', 'variant_allele': 'G', 'impact': 'MODIFIER', 'consequence_terms': ['downstream_gene_variant'], 'biotype': 'retained_intron', 'strand': 1}, {'transcript_id'

In [131]:
variant_vep = decoded[0]
variant_vep

{'strand': 1,
 'seq_region_name': '14',
 'end': 20357524,
 'transcript_consequences': [{'transcript_id': 'ENST00000250416',
   'gene_id': 'ENSG00000129484',
   'gene_symbol': 'PARP2',
   'variant_allele': 'G',
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'biotype': 'protein_coding',
   'strand': 1,
   'impact': 'LOW'},
  {'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'LOW',
   'consequence_terms': ['splice_donor_region_variant', 'intron_variant'],
   'strand': 1,
   'biotype': 'protein_coding',
   'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000429687',
   'gene_symbol': 'PARP2'},
  {'gene_id': 'ENSG00000129484',
   'transcript_id': 'ENST00000527384',
   'gene_symbol': 'PARP2',
   'distance': 1569,
   'hgnc_id': 'HGNC:272',
   'gene_symbol_source': 'HGNC',
   'variant_allele': 'G',
   'impact': 'MODIFIER',
   'consequence_terms': ['dow